In [9]:
import os
import glob
import sys
import warnings
import importlib.util
import subprocess

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

print("Python version:", sys.version)
print("\nFiles in current directory:")
for f in sorted(os.listdir(".")):
    print(f)

print("\nCSV files found:")
csv_files = sorted(glob.glob("*.csv"))
for f in csv_files:
    print(f)

Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]

Files in current directory:
.git
Catboost.ipynb
README.md
preprocessing.ipynb
sample_submission.csv
test.csv
train.csv

CSV files found:
sample_submission.csv
test.csv
train.csv


In [10]:
if importlib.util.find_spec("catboost") is None:
    print("CatBoost not found. Installing CatBoost...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "catboost", "-q"])
    print("CatBoost installation finished.")
else:
    print("CatBoost is already installed.")

CatBoost is already installed.


In [11]:
try:
    from catboost import CatBoostClassifier, Pool
    print("CatBoost imported successfully.")
except Exception as e:
    print("CatBoost import failed.")
    print("Error:", e)
    print("Please restart the notebook after CatBoost installation.")
    raise

CatBoost imported successfully.


In [12]:
def find_first_file(patterns):
    for pattern in patterns:
        matches = sorted(glob.glob(pattern))
        if matches:
            return matches[0]
    return None

train_path = find_first_file(["train.csv", "*train*.csv", "*Train*.csv"])
test_path = find_first_file(["test.csv", "*test*.csv", "*Test*.csv"])
sample_path = find_first_file([
    "sample_submission.csv",
    "*sample*submission*.csv",
    "*sample*.csv"
])

print("Selected train file:", train_path)
print("Selected test file:", test_path)
print("Selected sample submission file:", sample_path)

if train_path is None:
    raise FileNotFoundError("Could not find the training CSV file.")
if test_path is None:
    raise FileNotFoundError("Could not find the test CSV file.")
if sample_path is None:
    raise FileNotFoundError("Could not find the sample submission CSV file.")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_sub.shape)

print("\nTrain head:")
print(train.head())

print("\nTest head:")
print(test.head())

print("\nSample submission head:")
print(sample_sub.head())

Selected train file: train.csv
Selected test file: test.csv
Selected sample submission file: sample_submission.csv

Train shape: (668665, 15)
Test shape: (286571, 14)
Sample submission shape: (286571, 2)

Train head:
   id  Age  Annual_Income_USD  Daily_Commute_km  Number_of_Cars_Owned  \
0   0   66            92887.0              23.4                     2   
1   1   38            30000.0               5.0                     1   
2   2   26            94389.0              36.8                     1   
3   3   66            73580.0              23.7                     2   
4   4   54            57898.0              50.8                     1   

   Charging_Stations_Near_Home  Charging_Stations_Near_Work  \
0                            3                            7   
1                            2                            2   
2                            8                           15   
3                            6                            9   
4                            

In [13]:
print("Train columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nSample submission columns:")
print(sample_sub.columns.tolist())

target_col = "Will_Buy_EV"

print("\nTarget column in train:", target_col in train.columns)
print("Target column in test:", target_col in test.columns)

if target_col in train.columns:
    print("\nTarget value counts:")
    print(train[target_col].value_counts(dropna=False))

    print("\nTarget percentage:")
    print(train[target_col].value_counts(normalize=True, dropna=False) * 100)

print("\nMissing values in train:")
print(train.isna().sum().sort_values(ascending=False).head(20))

print("\nMissing values in test:")
print(test.isna().sum().sort_values(ascending=False).head(20))

Train columns:
['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV']

Test columns:
['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

Sample submission columns:
['id', 'Will_Buy_EV']

Target column in train: True
Target column in test: False

Target value counts:
Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64

Target percentage:
Will_Buy_EV
No     82.5355
Yes    17.4645
Name: proportion, dtype: float64

Missing values in train:
id                             0
Age                            0
A

In [14]:
target_col = "Will_Buy_EV"

if target_col not in train.columns:
    raise ValueError(f"Target column '{target_col}' was not found in train.")

train[target_col] = train[target_col].astype(str).str.strip().str.lower()

valid_targets = {"yes", "no"}
unknown_targets = sorted(set(train[target_col].unique()) - valid_targets)

if unknown_targets:
    raise ValueError(f"Unknown target values found: {unknown_targets}")

y = train[target_col].map({"yes": 1, "no": 0})

if y.isna().any():
    raise ValueError("Target mapping failed. Check target values.")

y = y.astype(int)

print("Target mapping complete.")
print("y value counts:")
print(y.value_counts())

print("\ny unique values:", sorted(y.unique().tolist()))

Target mapping complete.
y value counts:
Will_Buy_EV
0    551886
1    116779
Name: count, dtype: int64

y unique values: [0, 1]


In [15]:
id_col = "id"

if id_col not in train.columns:
    id_col = train.columns[0]
    print(f"'id' not found in train. Using first train column as id: {id_col}")
else:
    print("Using id column from train:", id_col)

if id_col not in test.columns:
    test_id_col = test.columns[0]
    print(f"'{id_col}' not found in test. Using first test column as test id: {test_id_col}")
else:
    test_id_col = id_col

expected_cat_features = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Range_Anxiety_Level",
    "Home_Charging_Possible",
    "Subsidy_Available"
]

cat_features = []

for c in expected_cat_features:
    if c in train.columns and c in test.columns:
        cat_features.append(c)
    else:
        print(f"Expected categorical column missing, skipping: {c}")

for c in train.columns:
    if c in [id_col, target_col]:
        continue
    if c in test.columns and (
        train[c].dtype == "object" or str(train[c].dtype) == "category"
    ):
        if c not in cat_features:
            cat_features.append(c)

print("\nCategorical features used by CatBoost:")
print(cat_features)

for c in cat_features:
    train[c] = train[c].astype(str).str.strip().str.lower()
    test[c] = test[c].astype(str).str.strip().str.lower()

    train[c] = train[c].replace({"nan": "unknown", "none": "unknown", "": "unknown"})
    test[c] = test[c].replace({"nan": "unknown", "none": "unknown", "": "unknown"})

    if train[c].isna().any() or test[c].isna().any():
        raise ValueError(f"Missing categorical values found in column: {c}")

print("\nCategorical columns normalized to strings and missing values handled.")

Using id column from train: id

Categorical features used by CatBoost:
['Gender', 'City_Type', 'Current_Car_Type', 'Range_Anxiety_Level', 'Home_Charging_Possible', 'Subsidy_Available']

Categorical columns normalized to strings and missing values handled.


In [16]:
feature_cols = [
    c for c in train.columns
    if c not in [id_col, target_col]
]

missing_in_test = [c for c in feature_cols if c not in test.columns]

if missing_in_test:
    raise ValueError(f"Missing feature columns in test: {missing_in_test}")

X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("\nFeature columns:")
print(feature_cols)

for c in feature_cols:
    if c in cat_features:
        X[c] = X[c].astype(str).fillna("unknown")
        X_test[c] = X_test[c].astype(str).fillna("unknown")
    else:
        if X[c].isna().any() or X_test[c].isna().any():
            median_value = X[c].median()
            X[c] = X[c].fillna(median_value)
            X_test[c] = X_test[c].fillna(median_value)

print("\nMissing values in X:", int(X.isna().sum().sum()))
print("Missing values in X_test:", int(X_test.isna().sum().sum()))

X shape: (668665, 13)
X_test shape: (286571, 13)

Feature columns:
['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

Missing values in X: 0
Missing values in X_test: 0


In [17]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)

print("\ny_train distribution:")
print(y_train.value_counts())

print("\ny_valid distribution:")
print(y_valid.value_counts())

X_train shape: (534932, 13)
X_valid shape: (133733, 13)
y_train shape: (534932,)
y_valid shape: (133733,)

y_train distribution:
Will_Buy_EV
0    441509
1     93423
Name: count, dtype: int64

y_valid distribution:
Will_Buy_EV
0    110377
1     23356
Name: count, dtype: int64


In [18]:
train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_features
)

valid_pool = Pool(
    data=X_valid,
    label=y_valid,
    cat_features=cat_features
)

test_pool = Pool(
    data=X_test,
    cat_features=cat_features
)

print("CatBoost Pool objects created successfully.")

CatBoost Pool objects created successfully.


In [19]:
model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    od_type="Iter",
    od_wait=150,
    auto_class_weights="Balanced",
    verbose=100,
    thread_count=-1
)

model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True
)

print("\nBest iteration:", model.get_best_iteration())
print("Best validation score:", model.get_best_score())

0:	test: 0.9289273	best: 0.9289273 (0)	total: 380ms	remaining: 18m 59s
100:	test: 0.9395551	best: 0.9395551 (100)	total: 19.3s	remaining: 9m 15s
200:	test: 0.9401033	best: 0.9401033 (200)	total: 39s	remaining: 9m 3s
300:	test: 0.9404832	best: 0.9404832 (300)	total: 1m	remaining: 8m 58s
400:	test: 0.9409094	best: 0.9409094 (400)	total: 1m 20s	remaining: 8m 42s
500:	test: 0.9411419	best: 0.9411419 (500)	total: 1m 39s	remaining: 8m 16s
600:	test: 0.9412692	best: 0.9412713 (596)	total: 1m 58s	remaining: 7m 53s
700:	test: 0.9413550	best: 0.9413550 (700)	total: 2m 17s	remaining: 7m 30s
800:	test: 0.9414569	best: 0.9414569 (800)	total: 2m 36s	remaining: 7m 9s
900:	test: 0.9414999	best: 0.9415024 (896)	total: 2m 55s	remaining: 6m 48s
1000:	test: 0.9415259	best: 0.9415259 (1000)	total: 3m 14s	remaining: 6m 28s
1100:	test: 0.9415611	best: 0.9415611 (1100)	total: 3m 33s	remaining: 6m 8s
1200:	test: 0.9415672	best: 0.9415672 (1200)	total: 3m 54s	remaining: 5m 51s
1300:	test: 0.9415670	best: 0.9415

In [20]:
valid_proba = model.predict_proba(valid_pool)[:, 1]
valid_pred = (valid_proba >= 0.5).astype(int)

roc_auc = roc_auc_score(y_valid, valid_proba)
accuracy = accuracy_score(y_valid, valid_pred)

print("Validation ROC-AUC:", roc_auc)
print("Validation Accuracy:", accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, valid_pred))

print("\nClassification Report:")
print(classification_report(y_valid, valid_pred, digits=4))

Validation ROC-AUC: 0.9415807264198256
Validation Accuracy: 0.854007612182483

Confusion Matrix:
[[92910 17467]
 [ 2057 21299]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9783    0.8418    0.9049    110377
           1     0.5494    0.9119    0.6857     23356

    accuracy                         0.8540    133733
   macro avg     0.7639    0.8768    0.7953    133733
weighted avg     0.9034    0.8540    0.8666    133733



In [21]:
print("Sample submission head:")
print(sample_sub.head())

print("\nSample submission columns:")
print(sample_sub.columns.tolist())

print("\nSample submission shape:")
print(sample_sub.shape)

print("\nSample submission dtypes:")
print(sample_sub.dtypes)

if len(sample_sub.columns) != 2:
    raise ValueError(
        "Expected sample_submission.csv to have exactly 2 columns: id and target."
    )

submission_id_col = sample_sub.columns[0]
submission_target_col = sample_sub.columns[1]

print("\nSubmission id column:", submission_id_col)
print("Submission target column:", submission_target_col)

Sample submission head:
       id  Will_Buy_EV
0  668665     0.174645
1  668666     0.174645
2  668667     0.174645
3  668668     0.174645
4  668669     0.174645

Sample submission columns:
['id', 'Will_Buy_EV']

Sample submission shape:
(286571, 2)

Sample submission dtypes:
id               int64
Will_Buy_EV    float64
dtype: object

Submission id column: id
Submission target column: Will_Buy_EV


In [22]:
test_proba = model.predict_proba(test_pool)[:, 1]

test_ids = test[test_id_col].values

pred_df = pd.DataFrame({
    test_id_col: test_ids,
    submission_target_col: test_proba
})

submission = sample_sub[[submission_id_col]].merge(
    pred_df,
    left_on=submission_id_col,
    right_on=test_id_col,
    how="left"
)

if test_id_col != submission_id_col and test_id_col in submission.columns:
    submission = submission.drop(columns=[test_id_col])

submission = submission[[submission_id_col, submission_target_col]]

print("Submission head:")
print(submission.head())

print("\nSubmission shape:")
print(submission.shape)

print("\nMissing predictions in submission:")
print(int(submission[submission_target_col].isna().sum()))

Submission head:
       id  Will_Buy_EV
0  668665     0.065658
1  668666     0.102935
2  668667     0.021878
3  668668     0.013339
4  668669     0.079488

Submission shape:
(286571, 2)

Missing predictions in submission:
0


In [23]:
print("Checking target mapping...")
if not set(y.unique()).issubset({0, 1}):
    raise ValueError("Target mapping is incorrect. y contains values other than 0 and 1.")
print("Target mapping OK.")

print("\nChecking missing values...")
print("X_train missing:", int(X_train.isna().sum().sum()))
print("X_valid missing:", int(X_valid.isna().sum().sum()))
print("X_test missing:", int(X_test.isna().sum().sum()))
print("Submission missing target:", int(submission[submission_target_col].isna().sum()))

if submission[submission_target_col].isna().any():
    raise ValueError("Submission contains missing predictions.")

print("\nChecking unknown categorical values in test compared with train...")
for c in cat_features:
    train_values = set(train[c].dropna().astype(str).unique())
    test_values = set(test[c].dropna().astype(str).unique())
    unknown_values = sorted(test_values - train_values)

    if unknown_values:
        print(f"{c}: {len(unknown_values)} unknown values. Examples: {unknown_values[:10]}")
    else:
        print(f"{c}: no unknown categorical values.")

print("\nChecking submission row count...")
print("Submission rows:", len(submission))
print("Sample submission rows:", len(sample_sub))

if len(submission) != len(sample_sub):
    raise ValueError("Submission row count does not match sample_submission.csv.")

print("\nChecking submission column names...")
print("Submission columns:", submission.columns.tolist())
print("Expected columns:", sample_sub.columns.tolist())

if list(submission.columns) != list(sample_sub.columns):
    raise ValueError("Submission column names do not match sample_submission.csv.")

print("\nAll validation checks passed.")

Checking target mapping...
Target mapping OK.

Checking missing values...
X_train missing: 0
X_valid missing: 0
X_test missing: 0
Submission missing target: 0

Checking unknown categorical values in test compared with train...
Gender: no unknown categorical values.
City_Type: no unknown categorical values.
Current_Car_Type: no unknown categorical values.
Range_Anxiety_Level: no unknown categorical values.
Home_Charging_Possible: no unknown categorical values.
Subsidy_Available: no unknown categorical values.

Checking submission row count...
Submission rows: 286571
Sample submission rows: 286571

Checking submission column names...
Submission columns: ['id', 'Will_Buy_EV']
Expected columns: ['id', 'Will_Buy_EV']

All validation checks passed.


In [24]:
submission.to_csv("submission.csv", index=False)

print("Saved submission.csv")
print("Final submission shape:", submission.shape)
print("\nFinal submission head:")
print(submission.head())

Saved submission.csv
Final submission shape: (286571, 2)

Final submission head:
       id  Will_Buy_EV
0  668665     0.065658
1  668666     0.102935
2  668667     0.021878
3  668668     0.013339
4  668669     0.079488


In [25]:
submission_check = pd.read_csv("submission.csv")

print("Reloaded submission.csv shape:", submission_check.shape)
print("\nReloaded columns:")
print(submission_check.columns.tolist())

print("\nReloaded head:")
print(submission_check.head())

print("\nReloaded missing values:")
print(submission_check.isna().sum())

Reloaded submission.csv shape: (286571, 2)

Reloaded columns:
['id', 'Will_Buy_EV']

Reloaded head:
       id  Will_Buy_EV
0  668665     0.065658
1  668666     0.102935
2  668667     0.021878
3  668668     0.013339
4  668669     0.079488

Reloaded missing values:
id             0
Will_Buy_EV    0
dtype: int64
